# Thinking in Probabilities

Companion notebook for the [Thinking in Probabilities](https://ml-viz.vercel.app/courses/probability-statistics/01-thinking-in-probabilities) lesson on ML Viz.

We'll make the abstract definitions concrete: simulate sample spaces, check the axioms by brute force, watch conditional probability "zoom in", and see the law of large numbers drag a running average toward the expectation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("dark_background")
plt.rcParams["figure.facecolor"] = "#0f1117"
plt.rcParams["axes.facecolor"] = "#1a1d27"
plt.rcParams["axes.edgecolor"] = "#2e3347"
plt.rcParams["grid.color"] = "#2e3347"

rng = np.random.default_rng(42)

## 1. Sample spaces and events

A sample space is just the set of possible outcomes. An event is a subset. Probability of an event = (in the equally-likely case) fraction of outcomes in it — and we can verify by simulation.

In [ ]:
omega = np.array([1, 2, 3, 4, 5, 6])          # sample space of one die roll
event_even = {2, 4, 6}                          # the event "roll is even"

rolls = rng.integers(1, 7, size=100_000)
p_even_sim = np.isin(rolls, list(event_even)).mean()
print(f"P(even) theoretical = {3/6:.3f},  simulated = {p_even_sim:.3f}")

## 2. The axioms, checked by brute force

Probabilities are non-negative, sum to 1 over the whole space, and add over disjoint events.

In [ ]:
p = np.array([(rolls == k).mean() for k in omega])
print("per-outcome probabilities:", p.round(3))
print("all non-negative:", (p >= 0).all())
print("sum over sample space:", p.sum().round(6))

# additivity over disjoint events: {1,2} and {5,6} share no outcomes
A, B = {1, 2}, {5, 6}
pA  = np.isin(rolls, list(A)).mean()
pB  = np.isin(rolls, list(B)).mean()
pAB = np.isin(rolls, list(A | B)).mean()
print(f"P(A)+P(B) = {pA+pB:.3f}  vs  P(A ∪ B) = {pAB:.3f}")

## 3. Conditional probability = zooming in

`P(A | B)` keeps only the worlds where B happened, then renormalizes. The product rule `P(A,B) = P(A|B) P(B)` follows immediately.

In [ ]:
# Roll two dice. A = "sum is 8", B = "first die shows 6"
d1 = rng.integers(1, 7, size=500_000)
d2 = rng.integers(1, 7, size=500_000)

A = (d1 + d2) == 8
B = d1 == 6

p_A_given_B = A[B].mean()           # zoom into the B-worlds
p_joint     = (A & B).mean()
p_B         = B.mean()

print(f"P(A|B) = {p_A_given_B:.4f}   (theory: 1/6 = {1/6:.4f})")
print(f"P(A|B)·P(B) = {p_A_given_B * p_B:.4f}  vs  P(A,B) = {p_joint:.4f}")

## 4. Expectation and the law of large numbers

E[X] = 3.5 for a fair die — a value the die never shows. Watch the running average of rolls converge to it.

In [ ]:
rolls = rng.integers(1, 7, size=10_000)
running_mean = np.cumsum(rolls) / np.arange(1, len(rolls) + 1)

plt.figure(figsize=(9, 4))
plt.plot(running_mean, color="#6366f1", lw=1.5, label="running average of rolls")
plt.axhline(3.5, color="#eab308", ls="--", label="E[X] = 3.5")
plt.xscale("log")
plt.xlabel("number of rolls (log scale)")
plt.ylabel("average value")
plt.title("Law of large numbers: the sample mean finds the expectation")
plt.legend()
plt.grid(alpha=0.4)
plt.show()

## 5. Variance: spread around the center

Compare a fair die with a '"loaded"' die that mostly shows 3 and 4 — same mean, very different spread.

In [ ]:
fair   = rng.integers(1, 7, size=100_000)
loaded = rng.choice([1, 2, 3, 4, 5, 6], p=[.05, .1, .35, .35, .1, .05], size=100_000)

for name, x in [("fair", fair), ("loaded", loaded)]:
    print(f"{name:>6}:  mean = {x.mean():.3f},  variance = {x.var():.3f}")

plt.figure(figsize=(9, 3.5))
for name, x, c in [("fair", fair, "#6366f1"), ("loaded", loaded, "#14b8a6")]:
    vals, counts = np.unique(x, return_counts=True)
    plt.bar(vals + (0.18 if name == "loaded" else -0.18), counts / len(x),
            width=0.34, color=c, label=name)
plt.xlabel("die face")
plt.ylabel("probability")
plt.title("Same expectation (3.5), different variance")
plt.legend()
plt.grid(alpha=0.4, axis="y")
plt.show()

**Next:** the [Probability Distributions](https://ml-viz.vercel.app/courses/probability-statistics/02-probability-distributions) lesson — the four named distributions (and their notebook) that ML reaches for daily.